# Understanding Memory-Mapped Files in nanoGPT

This notebook explores how `np.memmap` works for efficient data loading.
The concept: map a file directly to memory addresses without loading it all into RAM.

In [13]:
import numpy as np
import os
import time
import torch

# Path to Shakespeare data (small, good for learning)
data_path = "../data/shakespeare/train.bin"

# Check if data exists
if not os.path.exists(data_path):
    print("Run: python data/shakespeare/prepare.py first")
else:
    file_size = os.path.getsize(data_path)
    print(f"File size: {file_size:,} bytes ({file_size/1024/1024:.2f} MB)")

File size: 603,932 bytes (0.58 MB)


## Method 1: Load entire file into RAM (traditional way)

In [14]:
# Traditional loading - reads ENTIRE file into RAM
t0 = time.time()
data_loaded = np.fromfile(data_path, dtype=np.uint16)
t1 = time.time()

print(f"np.fromfile() - Full load")
print(f"  Time: {(t1-t0)*1000:.2f} ms")
print(f"  Shape: {data_loaded.shape}")
print(f"  RAM used: {data_loaded.nbytes:,} bytes")
print(f"  First 10 tokens: {data_loaded[:10]}")

np.fromfile() - Full load
  Time: 0.49 ms
  Shape: (301966,)
  RAM used: 603,932 bytes
  First 10 tokens: [ 5962 22307    25   198  8421   356  5120   597  2252    11]


## Method 2: Memory-mapped file (nanoGPT's approach)

Memory mapping creates a "view" into the file. The OS handles loading pages on-demand.

In [15]:
# Memory-mapped loading - does NOT read file into RAM yet
t0 = time.time()
data_mmap = np.memmap(data_path, dtype=np.uint16, mode='r')
t1 = time.time()

print(f"np.memmap() - Memory-mapped")
print(f"  Time: {(t1-t0)*1000:.2f} ms")  # Should be ~instant
print(f"  Shape: {data_mmap.shape}")
print(f"  File on disk, not in RAM yet!")

np.memmap() - Memory-mapped
  Time: 5.49 ms
  Shape: (301966,)
  File on disk, not in RAM yet!


In [16]:
# Accessing data triggers page loading - only accessed portion gets loaded into RAM
print("Accessing first 10 tokens:")
print(f"  data_mmap[:10] = {data_mmap[:10]}")

print("\nRandom access (like training does):")
block_size = 1024
batch_size = 4
ix = torch.randint(len(data_mmap) - block_size, (batch_size,))
print(f"  Random indices: {ix.tolist()}")
for i in ix:
    chunk = data_mmap[i:i+block_size]
    print(f"  data_mmap[{i}:{i+block_size}] -> shape {chunk.shape}")

Accessing first 10 tokens:
  data_mmap[:10] = [ 5962 22307    25   198  8421   356  5120   597  2252    11]

Random access (like training does):
  Random indices: [27520, 2259, 33385, 233735]
  data_mmap[27520:28544] -> shape (1024,)
  data_mmap[2259:3283] -> shape (1024,)
  data_mmap[33385:34409] -> shape (1024,)
  data_mmap[233735:234759] -> shape (1024,)


## Understanding the binary format

Each token is stored as `uint16` (2 bytes). GPT-2 vocab size is 50,257 which fits in uint16 (max 65,535).

In [17]:
# Look at raw bytes vs interpreted values
with open(data_path, 'rb') as f:
    raw_bytes = f.read(20)  # First 20 bytes = 10 uint16 values

print("Raw bytes (hex):")
print(f"  {raw_bytes.hex()}")

print("\nInterpreted as uint16 (little-endian):")
for i in range(0, 20, 2):
    val = int.from_bytes(raw_bytes[i:i+2], 'little')
    print(f"  bytes[{i}:{i+2}] = {raw_bytes[i:i+2].hex()} -> {val}")

print("\nSame via memmap:")
print(f"  {data_mmap[:10]}")

Raw bytes (hex):
  4a1723571900c600e520640100145502cc080b00

Interpreted as uint16 (little-endian):
  bytes[0:2] = 4a17 -> 5962
  bytes[2:4] = 2357 -> 22307
  bytes[4:6] = 1900 -> 25
  bytes[6:8] = c600 -> 198
  bytes[8:10] = e520 -> 8421
  bytes[10:12] = 6401 -> 356
  bytes[12:14] = 0014 -> 5120
  bytes[14:16] = 5502 -> 597
  bytes[16:18] = cc08 -> 2252
  bytes[18:20] = 0b00 -> 11

Same via memmap:
  [ 5962 22307    25   198  8421   356  5120   597  2252    11]


In [18]:
# Decode tokens back to text
import tiktoken
enc = tiktoken.get_encoding("gpt2")

# Decode first 100 tokens
first_tokens = data_mmap[:100].tolist()
decoded = enc.decode(first_tokens)
print("First 100 tokens decoded:")
print(decoded)
print(f"\n(Token IDs: {first_tokens[:20]}...)")

First 100 tokens decoded:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we

(Token IDs: [5962, 22307, 25, 198, 8421, 356, 5120, 597, 2252, 11, 3285, 502, 2740, 13, 198, 198, 3237, 25, 198, 5248]...)


## How nanoGPT uses this in training

The `get_batch()` function creates input (x) and target (y) pairs by shifting by 1 position.

In [19]:
# Simulate get_batch() from train.py
def get_batch(data, block_size=8, batch_size=4):
    """
    data: memory-mapped array of token IDs
    block_size: context length (how many tokens model sees)
    batch_size: number of sequences per batch
    
    Returns:
        x: input tokens  [batch_size, block_size]
        y: target tokens [batch_size, block_size] (shifted by 1)
    """
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    return x, y

# Demo
x, y = get_batch(data_mmap, block_size=8, batch_size=2)

print("Input (x) - what model sees:")
print(x)
print(f"\nTarget (y) - what model predicts:")
print(y)

print("\n--- Notice: y is x shifted by 1 position ---")
print(f"x[0]: {x[0].tolist()}")
print(f"y[0]: {y[0].tolist()}")

print("\nDecoded:")
print(f"x[0]: '{enc.decode(x[0].tolist())}'")
print(f"y[0]: '{enc.decode(y[0].tolist())}'")

Input (x) - what model sees:
tensor([[ 9838,    11,   534, 43989,   481,   307,   994,   379],
        [  198,  6943,  1062,   395,   540,  1918,    11,   416]])

Target (y) - what model predicts:
tensor([[   11,   534, 43989,   481,   307,   994,   379,  1755],
        [ 6943,  1062,   395,   540,  1918,    11,   416, 17903]])

--- Notice: y is x shifted by 1 position ---
x[0]: [9838, 11, 534, 43989, 481, 307, 994, 379]
y[0]: [11, 534, 43989, 481, 307, 994, 379, 1755]

Decoded:
x[0]: ' ye, your Romeo will be here at'
y[0]: ', your Romeo will be here at night'


## Memory stats comparison

In [20]:
# Compare memory footprint
print("=== Memory Comparison ===\n")

file_size = os.path.getsize(data_path)
n_tokens = len(data_mmap)

print(f"File: {data_path}")
print(f"  Size on disk: {file_size:,} bytes ({file_size/1024/1024:.2f} MB)")
print(f"  Total tokens: {n_tokens:,}")
print(f"  Bytes per token: {file_size/n_tokens:.1f} (uint16 = 2 bytes)")

print(f"\nnp.fromfile (full load):")
print(f"  RAM used: {data_loaded.nbytes:,} bytes (entire file)")

print(f"\nnp.memmap:")
print(f"  RAM used: ~0 bytes initially")
print(f"  Pages loaded on-demand (typically 4KB each)")

# Calculate what training actually loads
block_size = 1024
batch_size = 12
bytes_per_batch = batch_size * block_size * 2  # 2 bytes per uint16
print(f"\nPer training batch (batch_size={batch_size}, block_size={block_size}):")
print(f"  Tokens accessed: {batch_size * block_size:,}")
print(f"  Bytes accessed: {bytes_per_batch:,} ({bytes_per_batch/1024:.1f} KB)")
print(f"  % of file: {bytes_per_batch/file_size*100:.4f}%")

=== Memory Comparison ===

File: ../data/shakespeare/train.bin
  Size on disk: 603,932 bytes (0.58 MB)
  Total tokens: 301,966
  Bytes per token: 2.0 (uint16 = 2 bytes)

np.fromfile (full load):
  RAM used: 603,932 bytes (entire file)

np.memmap:
  RAM used: ~0 bytes initially
  Pages loaded on-demand (typically 4KB each)

Per training batch (batch_size=12, block_size=1024):
  Tokens accessed: 12,288
  Bytes accessed: 24,576 (24.0 KB)
  % of file: 4.0693%


## OpenWebText: mmap shines with large files

The real power of memmap shows with large datasets. OpenWebText is ~9GB - we can read any chunk instantly without loading the whole file.

In [21]:
# OpenWebText - large dataset (~9GB)
owt_path = "../data/openwebtext/train.bin"

if not os.path.exists(owt_path):
    print("OpenWebText not prepared. Run: python data/openwebtext/prepare.py")
else:
    owt_size = os.path.getsize(owt_path)
    print(f"OpenWebText file size: {owt_size:,} bytes ({owt_size/1024/1024/1024:.2f} GB)")
    
    # Memory-map it (instant, no RAM used)
    t0 = time.time()
    owt_data = np.memmap(owt_path, dtype=np.uint16, mode='r')
    t1 = time.time()
    print(f"mmap() time: {(t1-t0)*1000:.2f} ms")
    print(f"Total tokens: {len(owt_data):,} ({len(owt_data)/1e9:.2f} billion)")

OpenWebText file size: 18,071,164,978 bytes (16.83 GB)
mmap() time: 5.07 ms
Total tokens: 9,035,582,489 (9.04 billion)


In [22]:
# Read arbitrary chunks from anywhere in the 9GB file - each read is instant!
if os.path.exists(owt_path):
    print("Reading arbitrary chunks from 9GB file:\n")
    
    # Define some positions spread across the file
    positions = [
        0,                          # Start
        len(owt_data) // 4,          # 25%
        len(owt_data) // 2,          # 50%
        3 * len(owt_data) // 4,      # 75%
        len(owt_data) - 1024,        # End
    ]
    
    for pos in positions:
        t0 = time.time()
        chunk = owt_data[pos:pos+100]
        t1 = time.time()
        
        # Decode to text
        text = enc.decode(chunk.tolist())[:80].replace('\n', '\\n')
        
        print(f"Position {pos:,} ({pos/len(owt_data)*100:.0f}%):")
        print(f"  Read time: {(t1-t0)*1000:.3f} ms")
        print(f"  Text: \"{text}...\"")
        print()

Reading arbitrary chunks from 9GB file:

Position 0 (0%):
  Read time: 0.011 ms
  Text: "About the author: OBACK2KENYA is a 68-year-old retired private detective who fre..."

Position 2,258,895,622 (25%):
  Read time: 0.002 ms
  Text: " initial report on his Inner Earth visit, a popular channeler of the “Galactic F..."

Position 4,517,791,244 (50%):
  Read time: 0.002 ms
  Text: " is always moving up or down.\n\nRather than trying to predict exactly when Bitcoi..."

Position 6,776,686,866 (75%):
  Read time: 0.015 ms
  Text: " for always being the last person to leave the bar,” he says. “And sometimes inv..."

Position 9,035,581,465 (100%):
  Read time: 0.001 ms
  Text: " heads, wondering why three-toed sloths don't just let loose in the canopy like ..."



In [23]:
# Simulate training batches - random access across entire 9GB file
if os.path.exists(owt_path):
    print("Simulating training: random batch sampling from 9GB file\n")
    
    block_size = 1024
    batch_size = 12
    n_batches = 10
    
    times = []
    for batch in range(n_batches):
        t0 = time.time()
        
        # Random positions anywhere in the file
        ix = torch.randint(len(owt_data) - block_size, (batch_size,))
        x = torch.stack([torch.from_numpy(owt_data[i:i+block_size].astype(np.int64)) for i in ix])
        y = torch.stack([torch.from_numpy(owt_data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
        
        t1 = time.time()
        times.append((t1-t0)*1000)
    
    print(f"Batch config: {batch_size} sequences × {block_size} tokens = {batch_size*block_size:,} tokens/batch")
    print(f"Batch times: {[f'{t:.2f}ms' for t in times]}")
    print(f"Average: {np.mean(times):.2f} ms per batch")
    print(f"\nKey insight: We read {batch_size*block_size*2/1024:.1f} KB from random locations in a 9GB file!")
    print(f"Without mmap, we'd need 9GB RAM. With mmap, OS loads only the pages we touch.")

Simulating training: random batch sampling from 9GB file

Batch config: 12 sequences × 1024 tokens = 12,288 tokens/batch
Batch times: ['3.34ms', '2.24ms', '2.33ms', '2.15ms', '1.96ms', '2.05ms', '2.08ms', '2.42ms', '1.98ms', '2.09ms']
Average: 2.26 ms per batch

Key insight: We read 24.0 KB from random locations in a 9GB file!
Without mmap, we'd need 9GB RAM. With mmap, OS loads only the pages we touch.


In [24]:
owt_data

memmap([ 8585,   262,  1772, ..., 13815,    13, 50256],
       shape=(9035582489,), dtype=uint16)